In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS sona_agent_lab.hr_tools;

STEP 1 : CREATING A FUNCTION

In [0]:
%sql
CREATE OR REPLACE FUNCTION
sona_agent_lab.hr_tools.get_leave_balance(
    employee_id STRING
)
RETURNS INT
LANGUAGE SQL
RETURN
    CASE
        WHEN employee_id = 'EMP101' THEN 12
        WHEN employee_id = 'EMP102' THEN 8
        WHEN employee_id = 'EMP103' THEN 15
        ELSE 0
    END;

In [0]:
%sql
SELECT sona_agent_lab.hr_tools.get_leave_balance('EMP102')
AS remaining_leaves;

In [0]:
%sql
DESCRIBE FUNCTION
sona_agent_lab.hr_tools.get_leave_balance;

PRACTICAL

These BELOW packages provide the UC function integration and Databricks LangChain components.

In [0]:
%pip install unitycatalog-langchain[databricks]
%pip install databricks-langchain


In [0]:
dbutils.library.restartPython()

In [0]:
%pip show langchain-core langchain langchain-community databricks-langchain unitycatalog-langchain

In [0]:
%pip install --upgrade "unitycatalog-langchain[databricks]==0.4.0"

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import UCFunctionToolkit

print("UCFunctionToolkit imported successfully!")

## Connect UCFunctionToolkit to our function

In [0]:
from databricks_langchain import UCFunctionToolkit

function_name = "sona_agent_lab.hr_tools.get_leave_balance"

toolkit = UCFunctionToolkit(
    function_names=[function_name]
)

tools = toolkit.tools

print(tools)

EXPLANTION FOR ABOVE CODE
Purpose: To expose Unity Catalog functions as tools that our LLM can use.

from databricks_langchain import UCFunctionToolkit

Imports the toolkit.

function_name = "sona_agent_lab.hr_tools.get_leave_balance"

Stores the full UC function name (catalog.schema.function).

toolkit = UCFunctionToolkit(
    function_names=[function_name]
)

Creates a toolkit and exposes the specified UC function as a tool.

tools = toolkit.tools

Retrieves the exposed tools.

print(tools)

Displays the tools.

### check whether the tool was actually created.

In [0]:
print("Number of tools:", len(tools))

for tool in tools:
    print("Tool name:", tool.name)
    print("Description:", tool.description)

Our Unity Catalog function has now been successfully exposed as a tool through UCFunctionToolkit

In [0]:
for tool in tools:
    print("Tool name:", tool.name)
    print("Tool description:", tool.description)
    print("Input schema:", tool.args_schema.model_json_schema())

Execute the Unity Catalog function through the tool CHECKING

In [0]:
result = tools[0].invoke({
    "employee_id": "EMP101"
})

print(result)

Understand what's happening

tools[0] – Selects our first tool from the toolkit.

.invoke() – Executes the tool.

employee_id – Supplies the required input.

EMP101 – The employee whose leave balance we want.

ChatDatabricks is the LangChain integration that lets our Python application communicate with a Databricks-hosted chat model.

In [0]:
from databricks_langchain import ChatDatabricks

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",
    temperature=0.1,
    max_tokens=300
)

In [0]:
response = llm.invoke("Explain Unity Catalog in Databricks in three simple points.")
print(response.content)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

for ep in w.serving_endpoints.list():
    print("Name:", ep.name)
    print("State:", ep.state)
    print("-" * 40)

bind the UC function to the LLM

In [0]:
llm_with_tools = llm.bind_tools(tools)

print("UC function successfully bound to LLM!")

In [0]:
response = llm_with_tools.invoke(
    "How many annual leave days does employee EMP101 have remaining? "
    "Use the get_leave_balance tool to find the answer."
)

print(response)

Why is it showing a tool call instead of an answer?

Imagine you're asking a receptionist:

You: How many leave days do I have?

The receptionist doesn't know your leave balance. So they ask the HR system to check it.

That's exactly what your LLM is doing!

LLM: Understands your question.

Tool call: Requests information from the HR function.

UC function: Checks the database/business logic and returns the balance.

LLM: Uses that result to answer you.

Your current output means:

"I understand that you want EMP101's leave balance. Please execute the get_leave_balance function with employee ID EMP101."

NOW NEXT STEP


In [0]:
tool_call = response.tool_calls[0]

tool_result = tools[0].invoke(tool_call["args"])

print(tool_result)

COMPLETE HUMAN READABLE ANS

In [0]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id=response.tool_calls[0]["id"]
)

final_response = llm_with_tools.invoke([
    {
        "role": "user",
        "content": "How many annual leave days does EMP101 have remaining?"
    },
    response,
    tool_message
])

print(final_response.content)